In [8]:
# Cell 1: Import necessary libraries

import os
import numpy as np
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser, RigidTransform, RotationMatrix,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, InverseKinematics, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization, 
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions, 
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder, 
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph, 
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
import mcubes
from functools import partial
import matplotlib.pyplot as plt
from ipywidgets import widgets
import quadprog

from pathlib import Path
import sys

from ciris_plant_visualizer import CIrisPlantVisualizer

In [9]:
!pip show drake

Name: drake
Version: 1.48.0
Summary: Model-based design and verification for robotics
Home-page: https://drake.mit.edu
Author: Drake Development Team
Author-email: drake-users@mit.edu
License: Various
Location: /home/julialopezgomez/miniforge3/envs/obmp/lib/python3.13/site-packages
Requires: matplotlib, Mosek, numpy, pydot, PyYAML
Required-by: 


In [10]:
# Cell 2: Set up the plant and scene graph, and initialize the CIrisPlantVisualizer
# builder = DiagramBuilder()
# plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
# parser = Parser(plant, scene_graph)
# parser.SetAutoRenaming(True)

print("Setting up the plant and scene graph...")

# Replace DiagramBuilder with RobotDiagramBuilder
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [ ]:
# Add the robot
# gripper = parser.AddModels(file_name="my_sdfs/wsg_2dof.sdf")[0]

print("Loading Panda robot models...")

# --- Add the Panda arm + hand ---
# panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]
panda_arm  = parser.AddModels(file_name="my_sdfs/panda_arm_locked_4dof.urdf")[0]
# panda_hand = parser.AddModels(file_name="my_sdfs/panda_hand_locked_3dof.urdf")[0]

print("Welding Panda arm and hand...")

# Weld arm base to world (identity)
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

# Weld hand to arm flange with X_PC: translation [0,0,0], RPY deg [0,0,-45]
X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),        # parent (P)
    plant.GetFrameByName("panda_hand", panda_hand),    # child  (C)
    X_8H)

print("Setting default finger joint positions...")

# Optional: set the default finger opening (each finger is a prismatic joint).
# 0.02 m on each finger → ~0.04 m total width. Adjust to taste.
for j in ["panda_finger_joint1"]:#, "panda_finger_joint2"]:
    plant.GetJointByName(j, panda_hand).set_default_translation(-0.024)
    

    


In [12]:
print("Adding the bottle cap and obstacles...")

cap = parser.AddModels(file_name="my_sdfs/bottle_cap.sdf")[0]
obstacle1 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
# obstacle2 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
obstacle3 = parser.AddModels("my_sdfs/obstacle.sdf")[0]

# Set welds
plant.WeldFrames(
    plant.world_frame(), 
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

# Weld the obstacle to the world frame (adjust pose as needed)
obstacle_pose1 = RigidTransform(RotationMatrix(), [0.51, 0.031, 0.01])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle1),
    obstacle_pose1)

# # obstacle_pose2 = RigidTransform(RotationMatrix(), [-0.025, 0.05, 0.01])  # Adjust position
# # plant.WeldFrames(
# #     plant.world_frame(),
# #     plant.GetFrameByName("obstacle_link", obstacle2),
# #     obstacle_pose2)

obstacle_pose3 = RigidTransform(RotationMatrix(), [0.467, -0.005, 0.01])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle3),
    obstacle_pose3)

Adding the bottle cap and obstacles...


<WeldJoint name='world_welds_to_obstacle_link' index=16 model_instance=6>

In [13]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())

# Cell 3: Initialize the CIrisPlantVisualizer
q_star = np.zeros(plant.num_positions())


print("Initialising CspaceFreePolytope...")

# The object we will use to perform our certification
cspace_free_polytope = CspaceFreePolytope(
    plant, 
    scene_graph,
    SeparatingPlaneOrder.kAffine,
    q_star)

print("Initializing CIrisPlantVisualizer...")

visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    cspace_free_polytope,
    viz_role=Role.kIllustration,
    # viz_role=Role.kProximity,
    allow_plus_3dof=True
)

print("Setting up the visualizer...")

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)


INFO:drake:Meshcat listening for connections at http://localhost:7002


Finalising plant...
Bodies: 18 Joints: 17
Frames: 18 Geometries: 94
Plant finalised.
Number of positions:  4
Initialising CspaceFreePolytope...
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized
Setting up the visualizer...


In [14]:

sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())

def report_collisions(q):
    plant.SetPositions(plant_context, q)
    query_object = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    inspector = scene_graph.model_inspector()

    pairs = query_object.ComputePointPairPenetration()
    if not pairs:
        print("No penetrations")
        return

    for pair in pairs:
        name_A = inspector.GetName(pair.id_A)
        name_B = inspector.GetName(pair.id_B)
        print(f"{name_A} <-> {name_B}, depth={pair.depth}")



out = widgets.Output()
display(out)

def handle_slider_change(change, idx):
    q[idx] = change["new"]
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    with out:
        out.clear_output(wait=True)
        
        report_collisions(q)
        print(f"{visualizer.check_collision_q_by_ik(q)} \t {q}", flush=True)
    
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

Output()

FloatSlider(value=0.0, description='q0', max=1.0, min=-1.0, step=0.02)

FloatSlider(value=0.0, description='q1', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q2', max=0.045, step=0.00045)

FloatSlider(value=0.0, description='q3', max=3.14, min=-3.14, step=0.06280000000000001)

In [16]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()


# active_indices = [
#     joint_position_index(plant, "panda_joint7", panda_arm),
#     joint_position_index(plant, "panda_finger_joint1", panda_hand),
#     joint_position_index(plant, "cap_to_base", cap)
# ]

# active_indices = list(active_indices)
# fixed_indices = [i for i in range(plant.num_positions()) if i not in active_indices]

# print("Active indices:", active_indices)
# print("Fixed indices:", fixed_indices)

for i, name in enumerate(plant.GetPositionNames()):
    # tag = "ACTIVE" if i in active_indices else "fixed"
    print(f"{i:2d}: {name:40s}")

 0: panda_panda_joint7_q                    
 1: panda_hand_panda_finger_joint1_x        
 2: panda_hand_panda_finger_joint2_x        
 3: bottle_cap_cap_to_base_q                


## Grasping Configuration checker

Hand frame must be located at a certain height range from the cap (and rotation, so a transform), fingers at an offset (also a range) and allow full rotation of gripper hand and cap

In [20]:
# Get panda_hand frame between 0.0105 m and 0.11 m above the cap base_link frame
# Fingers should be open between 0.024 and 0.025 m and -0.024 and -0.025 m
# Get a function that checks whether these constraints are satisfied

def check_grasp_constraints(q):
    old_q = plant.GetPositions(plant_context)
    plant.SetPositions(plant_context, q)
    # diagram.ForcedPublish(diagram_context)
    
    # Get the current pose of the hand frame
    X_WE = plant.CalcRelativeTransform(plant_context, plant.world_frame(), E)
    X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)
    
    # Compute the relative transform from Cap to E
    X_CapE = X_WCap.inverse().multiply(X_WE)
    
    z_height = X_CapE.translation()[2]
    
    right_finger_joint = plant.GetJointByName("panda_finger_joint1", panda_hand)
    left_finger_joint  = plant.GetJointByName("panda_finger_joint2", panda_hand) # Uncomment this line if you want to check the left finger position as well
    
    right_finger_pos = right_finger_joint.get_translation(plant_context)
    left_finger_pos  = left_finger_joint.get_translation(plant_context) # Uncomment this line if you want to check the left finger position as well
    
    # Check height constraint
    height_ok = 0.0105 <= z_height <= 0.11
    
    # Check finger opening constraints
    fingers_ok = (-0.025 <= right_finger_pos <= -0.024) and (0.024 <= left_finger_pos <= 0.025)
    
    # Restore old q
    plant.SetPositions(plant_context, old_q)
    diagram.ForcedPublish(diagram_context)
    
    print(f"Height: {z_height:.4f} m, Right Finger: {right_finger_pos:.4f} m, Left Finger: {left_finger_pos:.4f} m, Height OK: {height_ok}, Fingers OK: {fingers_ok}")
    
    return height_ok and fingers_ok, z_height, right_finger_pos #, left_finger_pos

In [21]:
# Build diagram/contexts as you already do
plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

# Frames and goal pose
E   = plant.GetFrameByName("panda_hand", panda_hand)   # tool frame
Cap = plant.GetFrameByName("base_link", cap)           # cap frame

# World poses
X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)

# Choose a grasp goal: 65 mm above cap origin (same orientation as Cap here)
# Rotate by 180 deg upside down
R_CapGoal = RotationMatrix(RollPitchYaw(np.pi, 0, 0))
X_CapGoal = RigidTransform(R_CapGoal, [0.0, 0.0, 0.105])
X_WG = X_WCap.multiply(X_CapGoal)  # desired world pose for the hand frame


q_grasp = np.array([1., -0.024, 0.024, 2.])

diagram.ForcedPublish(diagram_context)

print("Checking grasp constraints for q_grasp:", check_grasp_constraints(q_grasp))

Height: 0.1050 m, Right Finger: -0.0240 m, Left Finger: 0.0240 m, Height OK: True, Fingers OK: True
Checking grasp constraints for q_grasp: (True, np.float64(0.10500008908741364), -0.024)


## 1.2. Setup Grasping and Placement Space

In [22]:
builder = visualizer.builder
plant = visualizer.plant
scene_graph = visualizer.scene_graph
q_star = visualizer.q_star
rat_fk = visualizer.rat_forward_kin
inspector = visualizer.model_inspector
diagram = visualizer.task_space_diagram
context = visualizer.task_space_diagram_context
cspace_free_polytope = visualizer.cspace_free_polytope

In [40]:
# Define the 8 corner points of the convex hull
x_bounds = [-1., 1.]
y_bounds = [-0.055, -0.024]
z_bounds = [0.024, 0.055]
w_bounds = [-3.14, 3.14]


# lower_joint_limits = np.array([x_bounds[0], y_bounds[0], z_bounds[0]])
# upper_joint_limits = np.array([x_bounds[1], y_bounds[1], z_bounds[1]])

lower_joint_limits = np.array([x_bounds[0], y_bounds[0], z_bounds[0], w_bounds[0]])
upper_joint_limits = np.array([x_bounds[1], y_bounds[1], z_bounds[1], w_bounds[1]])

y_bounds_grasp = [-0.025, -0.024]

z_bounds_grasp = [0.024, 0.025]

# Generate all corner points
# placement_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds for z in z_bounds])
# grasp_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds for z in z_bounds_grasp])
###### For 2 dimensional example:
# placement_points = np.array([[x, y] for x in x_bounds for y in z_bounds])
# grasp_points = np.array([[x, y] for x in x_bounds for y in z_bounds_grasp])

placement_points = np.array([[x, y, z, w] for x in x_bounds for y in y_bounds for z in z_bounds for w in w_bounds])
grasp_points = np.array([[x, y, z, w] for x in x_bounds for y in y_bounds_grasp for z in z_bounds_grasp for w in w_bounds])

# Compute the convex hull
placement_hull = ConvexHull(placement_points)
grasp_hull = ConvexHull(grasp_points)

# Convert ConvexHull to HPolyhedron
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)

placement_polytope = convex_hull_to_hpolyhedron(placement_hull)
grasp_polytope = convex_hull_to_hpolyhedron(grasp_hull)

In [41]:
print(visualizer.q_lower_limits)
print(visualizer.q_upper_limits)

[-1.    -0.045  0.    -3.14 ]
[1.    0.    0.045 3.14 ]


# Manipulation Planning Class

In [49]:
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile


class ManipulationPlanner():

    def __init__(self, 
            visualizer: CIrisPlantVisualizer,
            CP: HPolyhedron,
            CG: HPolyhedron,
            max_grasps: int = 20,
            model_instances: list = None,
            gripper_dim: int = None,
            cs_free: list[HPolyhedron] = None
        ):
        self.builder = visualizer.builder
        self.plant = visualizer.plant
        self.plant_context = visualizer.plant_context
        self.scene_graph = visualizer.scene_graph
        self.q_star = visualizer.q_star
        self.rat_fk = visualizer.rat_forward_kin
        self.inspector = visualizer.model_inspector
        self.diagram = visualizer.task_space_diagram
        self.diagram_context = visualizer.task_space_diagram_context
        self.lower_joint_limits = visualizer.q_lower_limits
        self.upper_joint_limits = visualizer.q_upper_limits
        self.visualize_cspace = visualizer.visualize_collision_constraint
        self.cspace_free_polytope = visualizer.cspace_free_polytope
        
        self.CP = CP
        self.CG = CG
        self.max_grasps = max_grasps

        self.model_instances = model_instances if model_instances is not None else [self.plant.GetModelInstanceByName("robot")]
        
        self.q_dim = self.plant.num_positions()
        self.gripper_dim = gripper_dim if gripper_dim is not None else self.q_dim
        
        
        self.cs_free = cs_free if cs_free is not None else self._generate_cfree()
    
        
        
        os.environ["MOSEKLM_LICENSE_FILE"] = "mosek.lic"
        with open(os.environ["MOSEKLM_LICENSE_FILE"], 'r') as f:
            contents = f.read()
            mosek_file_not_empty = contents != ''
                    
        assert mosek_file_not_empty, "Mosek license file is empty. Please ensure you have a valid license file at the path specified by MOSEKLM_LICENSE_FILE environment variable."
        assert MosekSolver().available(), "Mosek solver is not available. Please ensure you have Mosek installed and properly configured."
        
    def set_max_grasps(self, max_grasps: int):
        self.max_grasps = max_grasps
    
    
    def compute_trajectory(self, x_init, x_goal, display=True):
        print("Finding path for minimum grasps")
        path = self._find_minimum_grasp_path(
                x_init = x_init,
                x_goal = x_goal,
                P = self.CP,
                G = self.CG,
                lower_joint_limits = self.lower_joint_limits,
                upper_joint_limits = self.upper_joint_limits,
                c_free_polytopes = self.cs_free
            )
        
        print("Path found. Generating trajectory")
        traj = self._generate_trajectory(path)
        
        
        print("Trajectory generated. Starting display")
        if display:
            self.display_trajectory(traj)
            
        return path, traj
        
    def display_trajectory(self, traj, meshcat=True, plotly=False):
        if meshcat:
            num_points = int((traj.end_time() - traj.start_time()) * 4000)
            for t in np.linspace(traj.start_time(), traj.end_time(), num_points):
                q = traj.value(t)
                # self.plant.GetJointByName("left_finger_sliding_joint", gripper).set_translation(self.plant_context, q[2])
                # # self.plant.GetJointByName("right_finger_sliding_joint", gripper).set_translation(self.plant_context, q[3])
                # self.plant.GetJointByName("base_revolute_joint", gripper).set_angle(self.plant_context, q[1])
                # self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[0])
                self.plant.GetJointByName("panda_finger_joint1", panda_hand).set_translation(self.plant_context, q[1][0])
                self.plant.GetJointByName("panda_joint7", panda_arm).set_angle(self.plant_context, q[0][0])
                self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[2][0])
                self.diagram.ForcedPublish(self.diagram_context)
                time.sleep(0.01)
        path = [traj.value(t) for t in np.linspace(traj.start_time(), traj.end_time(), 100)]
        path_q = [q.ravel() for q in path] # Convert to list of numpy arrays
        if plotly:
            self.visualize_cspace(factor=1, num_points=30, paths=[path_q], filled_polytopes=self.cs_free)
        return path_q
    
    def _generate_trajectory(self, path):
        trajs = []
        print("Generating trajectory from:")
        for q0, q1 in zip(path[:-1], path[1:]):
            print("From\t{}\tto\t{}".format(q0, q1))
            traj, result = self._generate_edge_trajectory(q0, q1)
            if not result.is_success():
                print("failed to generate edge trajectory from {} to {}".format(q0, q1))
                return None
            trajs.append(traj)
        return CompositeTrajectory.AlignAndConcatenate(trajs)
        
        
    def _generate_edge_trajectory(self, x_init, x_goal, path_length_weight=None):
        trajopt = GcsTrajectoryOptimization(self.plant.num_positions())
        gcs_regions = trajopt.AddRegions(self.cs_free, order=1, h_min=0.01)
        source = trajopt.AddRegions([Point(x_init)], order=0)
        target = trajopt.AddRegions([Point(x_goal)], order=0)
        trajopt.AddEdges(source, gcs_regions)
        trajopt.AddEdges(gcs_regions, target)
        trajopt.AddPathLengthCost(path_length_weight if path_length_weight is not None else 1.0)
        options = GraphOfConvexSetsOptions()
        [traj, result] = trajopt.SolvePath(source, target, options)
        print(f"result.is_success() = {result.is_success()}")
        print(f"result.get_solution_result() = {result.get_solution_result()}")
        print(f"result.get_solver_id().name() = {result.get_solver_id().name()}")
        
        
        # print("\n--- GCS RESULT ---")
        # print("success:", result.is_success())
        # print("solution result:", result.get_solution_result())
        # print("solver:", result.get_solver_id().name())

        # # try:
        # #     details = result.get_solver_details()
        # #     print("solver details:", details)

        # #     for field in [
        # #         "rescode",
        # #         "solution_status",
        # #         "optimizer_time",
        # #     ]:
        # #         if hasattr(details, field):
        # #             print(f"{field}:", getattr(details, field))
        # # except Exception as e:
        # #     print("Could not read solver details:", e)

        # print("------------------\n")
        
        return traj, result        
        
    
    def _find_minimum_grasp_path(
            self,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray, 
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
            
        for i in range(self.max_grasps):
            if i % 10 == 0 and i > 0:
                print(f"Trying with {i} grasps")
            path = self._solve_for_n_grasps_CC(
                i,
                x_init,
                x_goal,
                P,
                G,
                lower_joint_limits,
                upper_joint_limits,
                c_free_polytopes
            )
            
            if path is not None:
                print(f"Found a solution with {i} grasps")
                return path  
        print(f"No solution found for less than {self.max_grasps} grasps")
    
    
    def _solve_for_n_grasps_CC(
            self,
            n_grasps: int, 
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray, 
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
        # Initialize the program
        prog = MathematicalProgram()
        n_points = 2 * n_grasps + 2 # each grasp has a grasp and a release point, plus the initial and goal points
        n_vars = self.q_dim * n_points # for each point of q_dim length, we have n_points*q_dim variables
        non_gripper_dim = self.q_dim - self.gripper_dim # number of non-gripper variables per point
        
        # Create decision variables
        x = prog.NewContinuousVariables(n_vars, "x")
        
        # 1. Cost function: Minimize sum of squared distances between consecutive points
        """ ||x_{i+1} - x_{i}||^2  equiv || Cx - c ||^2  == 0 """
        
        c = np.zeros((n_vars - self.q_dim, 1)) 
        C = np.zeros((n_vars - self.q_dim, n_vars))
        C[:, self.q_dim:] = np.eye(n_vars - self.q_dim)
        C[:, :-self.q_dim] -= np.eye(n_vars - self.q_dim)
        
        Q, b = self._to_quadratic_form(C, c)
        Q += np.eye(Q.shape[1]) * 1e-6  # Add a small value to the diagonal to make Q positive definite
        
        prog.AddQuadraticCost(Q=Q, b=b, vars=x)
        
        # 2. Initial and goal constraints
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_init.flatten(), x[:self.q_dim])
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_goal.flatten(), x[-self.q_dim:])
        
        
        ###### Placement and Grasping Constraints should be adapted task specific #####
        
        # 3. Placement constraints (equality) -> Cap remains constant in transit paths
        """x_cap_{i+1} - x_cap_{i} == 0 for all even i, odd i+1"""
        for i in range(n_grasps + 1):
            idx = 2*self.q_dim*i # points to first variable of x_i for even i's
            idx1_cap = idx + self.q_dim - 1 # points to last variable of x_i (cap orientation)
            idx2_cap = idx1_cap + self.q_dim # points to last variable of x_{i+1} (cap orientation)
            prog.AddLinearEqualityConstraint(x[idx2_cap] - x[idx1_cap] == 0)
        
        # 4. Grasp constraints (equality between gripper and cap orientations)
        """x_cap_i - x_wrist_i - x_cap_{i+1} + x_wrist_{i+1} == 0 for odd i, even i+1"""
        for i in range(n_grasps): 
            
            idxp1 = 2*self.q_dim*(i+1) # this idx points to first variable of x_i+1
            idx   = idxp1 - self.q_dim # this idx points to first variable of x_i
            
            idxp1_wrist = idxp1 + non_gripper_dim
            idxp1_cap = idxp1 + self.q_dim - 1
            idx_wrist = idx + non_gripper_dim
            idx_cap = idx + self.q_dim - 1 # equiv to idxp1 - 1
            
            prog.AddLinearEqualityConstraint(x[idx_cap] - x[idx_wrist] - x[idxp1_cap] + x[idxp1_wrist] == 0)
        
        ###### end of manual constraints
        
        
        # 5. Inequality Constraints (Placement hull)
        for i in range(n_points):
            prog.AddLinearConstraint(
                P.A(),  # Coefficient matrix
                -np.inf * np.ones_like(P.b()),  # Lower bound
                P.b(),  # Upper bound
                x[self.q_dim*i:self.q_dim*(i+1)]
            )
        # 6. Inequality Constraints (Grasp hull) 
        for i in range(n_points - 2):
            prog.AddLinearConstraint(
                G.A(),
                -np.inf * np.ones_like(G.b()),
                G.b(),
                x[self.q_dim*(i+1):self.q_dim*(i+2)]
            )
        
        # 7. Joint limits (inequality)
        prog.AddBoundingBoxConstraint(
            np.tile(lower_joint_limits, n_points),
            np.tile(upper_joint_limits, n_points),
            x
        )
        
        # 7. Collision-free polytope constraints (MIP)
        # Ensure all points are contained in at least one polytope
        
        M = 1e6  # Big-M constant (adjust based on problem scale)
        for i in range(n_points):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            
            # Creates one bineary 0-1 variable for each polytope
            z_i = prog.NewBinaryVariables(len(c_free_polytopes), name=f"z_{i}")
            
            # Each point x_i must be contained in at least one polytope
            prog.AddLinearConstraint(sum(z_i) >= 1)
            
            for j, poly in enumerate(c_free_polytopes):
                A_j = poly.A()
                b_j = poly.b()
                z_j = z_i[j]
                
                # Add constraints for each row of the polytope
                for k in range(A_j.shape[0]):
                    # Construct coefficient matrix [A_j_row | M]
                    coeffs = np.hstack([A_j[k], M])
                    # Combine x_i and z_j into variable vector
                    vars = np.hstack([x_i, [z_j]])
                    # A_j @ x_i + M * z[j] <= b_j + M
                    prog.AddLinearConstraint(
                        coeffs,
                        -np.inf,
                        b_j[k] + M,
                        vars
                    )
                
        # 8. Connected components constraints
        connected_components = self._compute_connected_components(P, G, c_free_polytopes)
        K = len(connected_components)
        
        # create binary variables for component membership (excluding init ang goal)
        z = {}
        for i in range (1, n_points-1): # exclude init and goal
            z[i] = prog.NewBinaryVariables(K, f"z_{i}")   
            # Each point must belong to exactly one component
            prog.AddLinearConstraint(sum(z[i]) == 1)
        
        # Consecutive points must belong to the same component, ignoring init and goal
        for i in range(1, n_points-2, 2):
            for k in range(K):
                prog.AddLinearConstraint(z[i][k] == z[i+1][k])
        
        # Enforce that each component contains at least one point
        for i in range(1, n_points-1):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            z_i = z[i]
            
            for k, comp in enumerate(connected_components):
                A_k = comp.A()
                b_k = comp.b()
                z_k = z_i[k]
                
                for j in range(A_k.shape[0]):
                    coeffs = np.hstack([A_k[j], M])
                    vars = np.hstack([x_i, [z_k]])
                    prog.AddLinearConstraint(coeffs, -np.inf, b_k[j] + M, vars)
        
        # Solve the problem
        solver = MosekSolver()
        result = solver.Solve(prog)
        
        if not result.is_success():
            return None
        
        return result.GetSolution(x).reshape(-1, self.q_dim)
    
    @staticmethod
    def _to_quadratic_form(C, c):
        return np.dot(C.T, C), -np.dot(C.T, c).flatten()
    
    def _compute_connected_components(
        self,
        placement: HPolyhedron,
        grasp: HPolyhedron,
        c_free: list[HPolyhedron]
    ) -> list[HPolyhedron]:
        """
        Computes convex connected components as:
        {placement ∩ grasp ∩ (union of c_free)}.
        Returns convex hulls of connected regions.
        """
        components = []
        
        # Compute intersections with each c_free polytope
        for poly in c_free:
            intersection = placement.Intersection(grasp).Intersection(poly)
            if not intersection.IsEmpty():
                components.append(intersection)
        
        # Merge overlapping components
        merged = []
        for comp in components:
            add_to_merged = False
            for m in merged:
                if comp.Intersection(m).IsEmpty():
                    continue
                # Merge overlapping components
                vertices_comp = visualizer.get_polytope_vertices(comp)
                vertices_m = visualizer.get_polytope_vertices(m)
                vertices = np.vstack([vertices_comp, vertices_m])
                merged_hull = ConvexHull(vertices)
                merged_poly = convex_hull_to_hpolyhedron(merged_hull)
                merged.remove(m)
                merged.append(merged_poly)
                add_to_merged = True
                break
            if not add_to_merged:
                merged.append(comp)
        
        return merged
    
    
    def _generate_cfree(self):
        generator = RandomGenerator(1234)
        checker = SceneGraphCollisionChecker(
            model=self.diagram,
            robot_model_instances=self.model_instances,
            edge_step_size=0.01,
        )
        options = IrisFromCliqueCoverOptions()
        options.num_points_per_visibility_round = 200
        options.coverage_termination_threshold = 0.99
        options.iris_options.configuration_space_margin = 0.00001
        # See https://github.com/RobotLocomotion/drake/issues/21343 -> If getting this issue reduce c-space margin above
        regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
        return regions
    
    

In [51]:
model_instances = [plant.GetModelInstanceByName("panda"), plant.GetModelInstanceByName("panda_hand"), plant.GetModelInstanceByName("bottle_cap")]



# #### TODO: THIS IS ONLY FOR TESTING REMOVE LATER ON (argument in class too)
regions = LoadIrisRegionsYamlFile(
    "data/cfree/cfree_4dof.yaml")
# )
# regions_dict_149 = LoadIrisRegionsYamlFile(
#     "cfree_drake_1_49.yaml"
# )

# old_regions = list(regions_dict_148.values())
# new_regions = list(regions_dict_149.values())
regions = list(regions.values())


planner = ManipulationPlanner(visualizer, placement_polytope, grasp_polytope, 50, model_instances=model_instances, cs_free=regions)

In [52]:
    # x_init = np.array([-3.14, -0.78, -0.03]) # [cap angle, gripper angle, finger translation]
    # x_goal = np.array([3.14, -0.78, -0.03])

# switch to gripper angle, finger translation, cap angle for planner
x_init = np.array([-0.72, -0.03, 0.03, -3.14]) # [gripper angle, finger translation, cap angle]
x_goal = np.array([-0.72, -0.03, 0.025, 0.])

print(visualizer.check_collision_q_by_ik(x_init))
print(visualizer.check_collision_q_by_ik(x_goal))
path, traj = planner.compute_trajectory(x_init, x_goal)

0.0
0.0
Finding path for minimum grasps
Found a solution with 6 grasps
Path found. Generating trajectory
Generating trajectory from:
From	[-0.72 -0.03  0.03 -3.14]	to	[ 0.44211646 -0.025       0.02499999 -3.14      ]


INFO:drake:Solved GCS shortest path using MOSEK with convex_relaxation=true and preprocessing=true and rounding.
INFO:drake:Found 5 unique paths, discarded 0 duplicate paths.
INFO:drake:Finished 5 rounding solutions with MOSEK.


result.is_success() = True
result.get_solution_result() = SolutionResult.kSolutionFound
result.get_solver_id().name() = MOSEK
From	[ 0.44211646 -0.025       0.02499999 -3.14      ]	to	[ 0.99999999 -0.02497405  0.02497192 -2.58211645]


INFO:drake:Solved GCS shortest path using MOSEK with convex_relaxation=true and preprocessing=true and rounding.
INFO:drake:Found 5 unique paths, discarded 9 duplicate paths.
INFO:drake:Finished 5 rounding solutions with MOSEK.


result.is_success() = True
result.get_solution_result() = SolutionResult.kSolutionFound
result.get_solver_id().name() = MOSEK
From	[ 0.99999999 -0.02497405  0.02497192 -2.58211645]	to	[ 0.45886421 -0.025       0.02499999 -2.58211645]


...

result.is_success() = True
result.get_solution_result() = SolutionResult.kSolutionFound
result.get_solver_id().name() = MOSEK
From	[ 0.38229817 -0.025       0.02492073 -0.25481193]	to	[ 0.6371101  -0.02499958  0.02494852  0.        ]


INFO:drake:Solved GCS shortest path using MOSEK with convex_relaxation=true and preprocessing=true and rounding.
INFO:drake:Found 5 unique paths, discarded 14 duplicate paths.
INFO:drake:Finished 5 rounding solutions with MOSEK.


result.is_success() = True
result.get_solution_result() = SolutionResult.kSolutionFound
result.get_solver_id().name() = MOSEK
From	[ 0.6371101  -0.02499958  0.02494852  0.        ]	to	[-0.72  -0.03   0.025  0.   ]


INFO:drake:Solved GCS shortest path using MOSEK with convex_relaxation=true and preprocessing=true and rounding.
INFO:drake:Found 5 unique paths, discarded 0 duplicate paths.
INFO:drake:Finished 5 rounding solutions with MOSEK.


result.is_success() = True
result.get_solution_result() = SolutionResult.kSolutionFound
result.get_solver_id().name() = MOSEK
Trajectory generated. Starting display


In [ ]:
planner.plant.SetPositions(planner.plant_context, x_init)
traj_path = planner.display_trajectory(traj, plotly=False)


In [50]:
from pydrake.geometry.optimization import SaveIrisRegionsYamlFile

SaveIrisRegionsYamlFile(
    "data/cfree/cfree_4dof.yaml",
    {
        f"region_{i:03d}": region
        for i, region in enumerate(planner.cs_free)
    },
)

# np.save(
#     "path_drake_1_49.npy",
#     path,
# )

In [54]:
SaveIrisRegionsYamlFile(
    "data/cfree/connected_components_4dof.yaml",
    {
        f"region_{i:03d}": region
        for i, region in enumerate(planner._compute_connected_components(
            planner.CP, planner.CG, planner.cs_free))
    },
)